# Práctica del módulo 3

Ya hemos visto los fundamentos de la construcción de modelos de regresión con redes neuronales en TensorFlow.

Vamos a subir un nivel y construir un modelo para un conjunto de datos más rico en características: Intentaremos predecir el coste del seguro médico para individuos en función de varios parámetros como `edad`, `sexo`, `IMC (BMI)`, `hijos`, `estado de fumador` y `región de residencia`.

Para hacerlo, utilizaremos el conjunto de datos de [Costes Médicos](https://www.kaggle.com/mirichoi0218/insurance) disponible públicamente en Kaggle y alojado en [https://fp.cloud.riberadeltajo.es/ia/insurance.csv](https://fp.cloud.riberadeltajo.es/ia/insurance.csv).



In [ ]:
import tensorflow as tf
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:

insurance = pd.read_csv("https://fp.cloud.riberadeltajo.es/ia/insurance.csv")


In [ ]:
insurance.head()

## Visualización y ELT (Extract, Load, Transform)

Vamos a tener que convertir las columnas no numéricas en números (porque una red neuronal no puede manejar entradas no numéricas).

Para hacerlo, utilizaremos el método `get_dummies()` de pandas.

Este método convierte variables categóricas (como las columnas de sexo, fumador y región) en variables numéricas utilizando codificación one-hot (one-hot encoding).


In [ ]:
# Convierte características categóricas en one-hot


In [ ]:
# crea X e y cos los target a partir de la columna charges 


In [ ]:
# muestra las primeras líneas de X

In [ ]:
# crea conjuntos de entrenamiento y test


In [ ]:
# creamos un modelo sencillo, dos capas de una neurona, loss=mae, optimizer=sgd,metricas=mae
# entrenalo por 100 épocas

Parece que nuestro modelo no aprende mucho: 
Intentaremos 3 cosas:
- Aumentar el número de capas (de 2 a 3).
- Aumentar el número de unidades en cada capa (excepto en la capa de salida).
- Cambiar el optimizador (de SGD a Adam).

In [ ]:
# crea insurance_model_2: añade dos capas, una de 100 unidades y otra de 10
# cambia optimizador a Adam
# entrena 100 épocas


In [ ]:
# evalua


¡Mucho mejor! Usar un modelo más grande y el optimizador Adam resulta en casi la mitad del error en comparación con el modelo anterior.

> 🔑 **Nota:** Para muchos problemas, el [optimizador Adam](https://www.tensorflow.org/api_docs/python/tf/keras/optimizers/Adam) es una excelente opción inicial. Consulta el punto de Andrei Karpathy "Adam es seguro" en [*A Recipe for Training Neural Networks*](http://karpathy.github.io/2019/04/25/recipe/) para más información.

Vamos a revisar las curvas de pérdida de nuestro modelo, deberíamos ver una tendencia descendente.


In [ ]:
# imprime la curva de pérdida


A partir de esto, parece que la pérdida (y el MAE) de nuestro modelo seguían disminuyendo (en nuestro caso, MAE y pérdida son lo mismo, por lo que las líneas en el gráfico se superponen).

Esto nos dice que la pérdida podría seguir bajando si intentamos entrenarlo por más tiempo.

> 🤔 **Pregunta:** ¿Cuánto tiempo deberíamos entrenar?

> Depende del problema en el que estés trabajando. A veces el entrenamiento no tomará mucho tiempo, otras veces tomará más de lo que esperas. Un método común es entrenar el modelo por mucho tiempo (por ejemplo, miles de épocas) pero configurarlo con un [callback de EarlyStopping](https://www.tensorflow.org/api_docs/python/tf/keras/callbacks/EarlyStopping) para que se detenga automáticamente cuando deje de mejorar. Veremos esto en otro módulo.

Vamos a entrenar el mismo modelo de arriba por un poco más de tiempo. Podemos hacerlo llamando nuevamente a `fit` en el modelo.


In [ ]:
# entrena más épocas


In [ ]:
# evalua de nuevo

¡Boom! Entrenando por 100 épocas adicionales, vemos una reducción de error de aproximadamente un 10%.

¿Cómo se ve el gráfico?

In [ ]:
# Visualiza la pérdida del model (curva de "loss" tras 200 épocas)


## Preprocesamiento de datos (normalización y estandarización)

Una práctica común al trabajar con redes neuronales es asegurarse de que todos los datos que se les pasan estén en el rango de 0 a 1.

Esta práctica se llama **escalado** (escalar todos los valores de su rango original, por ejemplo, de entre 0 y 100,000 para que estén entre 0 y 1).

Hay otro proceso llamado **estandarización**, que convierte todos tus datos a una varianza unitaria y una media de 0.

Estas dos prácticas suelen formar parte de una tubería de preprocesamiento (una serie de funciones para preparar tus datos para su uso con redes neuronales).

Sabiendo esto, algunos de los principales pasos que tomarás para preprocesar tus datos para una red neuronal incluyen:
* Convertir todos tus datos en números (una red neuronal no puede manejar cadenas de texto).
* Asegurarse de que tus datos tengan la forma correcta (verificar las formas de entrada y salida).
* [**Escalado de características**](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-scaler):
    
   - **Escalar**: Ajusta los datos a un rango específico (ej. 0-1), manteniendo la distribución.
   - **Estandarizar**: Ajusta los datos para que tengan media 0 y desviación estándar 1.
   - **Normalizar**: Ajusta los datos en una escala uniforme, a menudo en un rango de 0 a 1 o como vector de unidad.
   
   * ¿Cuál deberías usar?
      * **Con redes neuronales tiendes a favorecer la normalización**, ya que prefieren valores entre 0 y 1 (esto lo verás especialmente en el procesamiento de imágenes), sin embargo, a menudo encontrarás que una red neuronal puede funcionar bastante bien con un escalado mínimo de características.

> 📖 **Recurso:** Para más información sobre el preprocesamiento de datos, te recomiendo leer los siguientes recursos:
* [Documentación de Scikit-Learn sobre el preprocesamiento de datos](https://scikit-learn.org/stable/modules/preprocessing.html#preprocessing-data).
* [Escalar, estandarizar o normalizar con Scikit-Learn por Jeff Hale](https://towardsdatascience.com/scale-standardize-or-normalize-with-scikit-learn-6ccc7d176a02).

Ya hemos convertido nuestros datos en números usando `get_dummies()`, veamos cómo los normalizaríamos también.


In [ ]:
insurance = pd.read_csv("https://fp.cloud.riberadeltajo.es/ia/insurance.csv")
insurance.head()

Ahora, al igual que antes, necesitamos transformar las columnas no numéricas en números y esta vez también normalizaremos las columnas numéricas con diferentes rangos (para asegurarnos de que todas estén entre 0 y 1).

Para hacer esto, vamos a usar algunas clases de Scikit-Learn:
* [`make_column_transformer`](https://scikit-learn.org/stable/modules/generated/sklearn.compose.make_column_transformer.html) - construir una función de preprocesamiento de datos de varios pasos para las siguientes transformaciones:
  * [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) - asegurarse de que todas las columnas numéricas estén normalizadas (entre 0 y 1).
  * [`OneHotEncoder`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) - aplicar codificación one hot a las columnas no numéricas.

Veámoslas en acción.


In [ ]:
from sklearn.compose import make_column_transformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder

# Crear un column transformer que nos ayude a normalizar/preprocesar nuestros datos
ct = make_column_transformer(
    (MinMaxScaler(), ["age", "bmi", "children"]), # ajustar todos los valores entre 0 y 1
    (OneHotEncoder(handle_unknown="ignore"), ["sex", "smoker", "region"])
)

# Creamos X e y 
X = insurance.drop("charges", axis=1)
y = insurance["charges"]

# Nos quedamos con un split de train y test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Entrenar el column transformer sólo con los datos de entrenamiento (hacerlo en test produciría "data leakage")
ct.fit(X_train)

# Transform training and test data with normalization (MinMaxScalar) and one hot encoding (OneHotEncoder)
X_train_normal = ct.transform(X_train)
X_test_normal = ct.transform(X_test)

In [ ]:
# Ejemplo de datos no normalizados y sin one-hot
X_train.loc[0]

In [ ]:
# Ejemplo con datos normalizados y one-hot
X_train_normal[0]

In [ ]:
# ¿Qué pinta tienen las shapes?
# Fíjate que la shape del normalizado/one-hot es más grande por las columnas extras
X_train_normal.shape, X_train.shape

Nuestros datos están normalizados y "numerizados", vamos a modelarlos, usaremos el mismo modelo que insurance_model_2.

In [ ]:
# genera insurance_model_3 como un clon de insurance_model_2 y entrénalo

In [ ]:
# evalua

In [ ]:
# compara con el modelo sin normalizar

A partir de esto, podemos ver que normalizar los datos resulta en un 10% menos de error usando el mismo modelo que cuando no se normalizan los datos.

Este es **uno de los principales beneficios de la normalización: menor tiempo de convergencia** (una manera elegante de decir que tu modelo llega a mejores resultados más rápido).

`insurance_model_2` podría haber alcanzado eventualmente los mismos resultados que `insurance_model_3` si lo hubiéramos dejado entrenar por más tiempo.

Además, los resultados podrían cambiar si alteráramos las arquitecturas de los modelos, por ejemplo, añadiendo más unidades ocultas por capa o más capas.

## Ahora tu
- Intenta mejorar los resultados de los modelos anteriores.
- Experimenta un poco más añadiendo capas 
- Añade neuronas a cada capa a ver si mejoras los resultados.
- Experimenta cambiando el parámetro learning_rate de Adam() 


